# 623 Stride: contract-driven five-capacity LSTM sweep

This notebook derives the active run ID, model points, revisions, input fields, and training configuration from the checked-in model contract. It first reuses the current run's verified single input archive in Google Drive. If none is available, one file chooser accepts a single prior/source `.colab_input.tar.gz`; multipart manifest plus parts remains an optional compatibility fallback. The archive's original name and SHA-256 are recorded, its contents are safely extracted, `SHA256SUMS` and the track validator are run before training, and every trainer's combined stdout/stderr is streamed and persisted in Drive. Output is one `.tar.gz` by default; multipart output is opt-in only if a single browser transfer is impractical.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import userdata
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), f'Select an A100 runtime, observed {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
DEVICE_NAME=torch.cuda.get_device_name(0)
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally:
    if pathlib.Path(ASKPASS).exists(): pathlib.Path(ASKPASS).unlink()
print(DEVICE_NAME,subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/train_and_offline_infer.py'
CONTRACT_SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/model_contract.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--describe-model-points'],text=True))
TRAINER_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
assert MODEL_CONTRACT==TRAINER_CONTRACT, 'trainer and model_contract descriptions differ'
for key in ('run_id','points','training_config','model_revision','decoder_revision','external_input_fields'): assert key in MODEL_CONTRACT,key
RUN_ID=MODEL_CONTRACT['run_id']; TRAINING=MODEL_CONTRACT['training_config']; TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']
assert RUN_ID and pathlib.Path(RUN_ID).name==RUN_ID and '\\' not in RUN_ID,RUN_ID
assert POLICY and pathlib.Path(POLICY).name==POLICY and '\\' not in POLICY,POLICY
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_{POLICY}/{RUN_ID}'
INPUT_DIR=f'/content/{RUN_ID}_colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'
TRANSFER_DIR=pathlib.Path(f'/content/{RUN_ID}_input_transfer')
CANONICAL_INPUT_NAME=f'{RUN_ID}.colab_input.tar.gz'
CANONICAL_INPUT_ARCHIVE=pathlib.Path(DRIVE_ROOT,CANONICAL_INPUT_NAME)
INPUT_PROVENANCE_PATH=pathlib.Path(DRIVE_ROOT,'input_archive_provenance.json')
os.makedirs(DRIVE_ROOT,exist_ok=True)
COMMON=f'{REPO}/formal_NN_training/common'; sys.path.insert(0,COMMON)
import split_colab_archive as transfer
assert transfer.MAX_PART_BYTES==90*1024*1024
def verify_and_extract_input(archive_path):
    archive_path=pathlib.Path(archive_path)
    if pathlib.Path(INPUT_DIR).exists(): shutil.rmtree(INPUT_DIR)
    transfer.safe_extract_tar_gz(archive_path,INPUT_DIR)
    verified=transfer.validate_sha256sums(INPUT_DIR)
    return {'name':archive_path.name,'size_bytes':archive_path.stat().st_size,'sha256':transfer.sha256_file(archive_path),'format':'tar+gzip'},verified
def save_provenance(payload):
    temporary=INPUT_PROVENANCE_PATH.with_name(f'.{INPUT_PROVENANCE_PATH.name}.writing')
    temporary.write_text(json.dumps(payload,indent=2,sort_keys=True)+'\n')
    os.replace(temporary,INPUT_PROVENANCE_PATH)
INPUT_ARCHIVE_RECORD=None; SHA256SUMS_VERIFIED=None
if CANONICAL_INPUT_ARCHIVE.is_file() and not CANONICAL_INPUT_ARCHIVE.is_symlink():
    try:
        canonical_record,SHA256SUMS_VERIFIED=verify_and_extract_input(CANONICAL_INPUT_ARCHIVE)
        if INPUT_PROVENANCE_PATH.is_file():
            INPUT_ARCHIVE_RECORD=json.loads(INPUT_PROVENANCE_PATH.read_text())
            assert INPUT_ARCHIVE_RECORD.get('schema')=='cache_arch.colab_input_provenance.v1'
            assert INPUT_ARCHIVE_RECORD['canonical_archive']==canonical_record
            source_record=INPUT_ARCHIVE_RECORD['source_archive']; source_name=source_record['name']
            assert pathlib.Path(source_name).name==source_name and '\\' not in source_name and source_name.endswith('.colab_input.tar.gz')
            assert source_record['size_bytes']==canonical_record['size_bytes'] and source_record['sha256']==canonical_record['sha256']
        else:
            INPUT_ARCHIVE_RECORD={'schema':'cache_arch.colab_input_provenance.v1','transfer_mode':'drive_cache_without_prior_sidecar','source_archive':canonical_record,'canonical_archive':canonical_record}
            save_provenance(INPUT_ARCHIVE_RECORD)
        print('reused verified Drive input',CANONICAL_INPUT_ARCHIVE)
    except Exception as exc:
        if pathlib.Path(INPUT_DIR).exists(): shutil.rmtree(INPUT_DIR)
        INPUT_ARCHIVE_RECORD=None; SHA256SUMS_VERIFIED=None
        print('Drive input cache was not reusable:',repr(exc),flush=True)
if INPUT_ARCHIVE_RECORD is None:
    print('Select one prior/source *.colab_input.tar.gz. Optional fallback: select one *.parts.json and all of its .part-* files in the same chooser.')
    uploaded=files.upload(); selected_names=set(uploaded)
    assert uploaded and all(pathlib.Path(name).name==name and '\\' not in name for name in uploaded),sorted(uploaded)
    if TRANSFER_DIR.exists(): shutil.rmtree(TRANSFER_DIR)
    TRANSFER_DIR.mkdir(parents=True)
    for upload_name,payload in uploaded.items(): pathlib.Path(TRANSFER_DIR,upload_name).write_bytes(payload)
    single_names=[name for name in selected_names if name.endswith('.colab_input.tar.gz')]
    manifest_names=[name for name in selected_names if name.endswith('.colab_input.tar.gz.parts.json')]
    if len(selected_names)==1 and len(single_names)==1:
        candidate=pathlib.Path(TRANSFER_DIR,single_names[0]); transfer_mode='single_upload'
    else:
        assert not single_names and len(manifest_names)==1,f'Choose one single archive, or one manifest plus exactly its parts; observed {sorted(selected_names)}'
        manifest_path=pathlib.Path(TRANSFER_DIR,manifest_names[0]); part_manifest=transfer.load_manifest(manifest_path)
        assert part_manifest['archive']['name'].endswith('.colab_input.tar.gz'),part_manifest['archive']
        assert manifest_path.name==f"{part_manifest['archive']['name']}.parts.json",manifest_path.name
        expected_names={manifest_path.name}|{part['name'] for part in part_manifest['parts']}
        assert selected_names==expected_names,(sorted(selected_names),sorted(expected_names))
        transfer.validate_parts(part_manifest,TRANSFER_DIR)
        candidate=pathlib.Path(TRANSFER_DIR,part_manifest['archive']['name'])
        transfer.reassemble_archive(manifest_path,TRANSFER_DIR,candidate,overwrite=True); transfer_mode='multipart_upload_fallback'
    source_record,SHA256SUMS_VERIFIED=verify_and_extract_input(candidate)
    temporary=CANONICAL_INPUT_ARCHIVE.with_name(f'.{CANONICAL_INPUT_ARCHIVE.name}.copying')
    if temporary.exists(): temporary.unlink()
    shutil.copyfile(candidate,temporary); os.replace(temporary,CANONICAL_INPUT_ARCHIVE)
    canonical_record={'name':CANONICAL_INPUT_NAME,'size_bytes':CANONICAL_INPUT_ARCHIVE.stat().st_size,'sha256':transfer.sha256_file(CANONICAL_INPUT_ARCHIVE),'format':'tar+gzip'}
    assert canonical_record['size_bytes']==source_record['size_bytes'] and canonical_record['sha256']==source_record['sha256']
    INPUT_ARCHIVE_RECORD={'schema':'cache_arch.colab_input_provenance.v1','transfer_mode':transfer_mode,'source_archive':source_record,'canonical_archive':canonical_record}
    save_provenance(INPUT_ARCHIVE_RECORD); del uploaded
assert INPUT_ARCHIVE_RECORD and SHA256SUMS_VERIFIED
print('verified input provenance',json.dumps(INPUT_ARCHIVE_RECORD,sort_keys=True),'payload_hashes',len(SHA256SUMS_VERIFIED))

In [ ]:
ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
PACKAGED_COLLECTION_MANIFEST=pathlib.Path(f'{INPUT_DIR}/collection_manifest.json'); assert PACKAGED_COLLECTION_MANIFEST.is_file()
VALIDATOR=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/validate_collected_inputs.py'
VALIDATION_DIR=pathlib.Path(DRIVE_ROOT,'input_validation'); VALIDATION_DIR.mkdir(parents=True,exist_ok=True)
VALIDATED_MANIFEST_PATH=VALIDATION_DIR/'validated_collection_manifest.json'
CHILD_ENV=os.environ.copy(); CHILD_ENV['PYTHONUNBUFFERED']='1'
subprocess.run([sys.executable,VALIDATOR,'--input-dir',INPUT_DIR,'--manifest-out',str(VALIDATED_MANIFEST_PATH)],check=True,env=CHILD_ENV)
collection_manifest=json.loads(VALIDATED_MANIFEST_PATH.read_text())
external_fields=MODEL_CONTRACT['external_input_fields']
expected_input={'status':'PASS','source_decision_effective_external_input':external_fields,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True}
bad={k:(collection_manifest.get(k),v) for k,v in expected_input.items() if collection_manifest.get(k)!=v}; assert not bad,bad
assert collection_manifest['training_runtime_fields']==external_fields==collection_manifest['inference_runtime_fields']
assert MODEL_CONTRACT.get('neural_role')=='standalone_direct_action_prefetcher'
assert MODEL_CONTRACT.get('normal_policy_outputs_used_as_model_inputs') is False
assert MODEL_CONTRACT.get('normal_policy_candidates_used_as_model_inputs') is False
assert MODEL_CONTRACT.get('normal_policy_private_state_used_as_model_inputs') is False
print('fresh input validation PASS',TRACE,POLICY,external_fields,VALIDATED_MANIFEST_PATH)

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
shutil.copy2(VALIDATED_MANIFEST_PATH,pathlib.Path(LOCAL_OUTPUT,'validated_collection_manifest.json'))
TRAIN_LOG_DIR=pathlib.Path(DRIVE_ROOT,'trainer_logs'); TRAIN_LOG_DIR.mkdir(parents=True,exist_ok=True)
def run_trainer_streamed(cmd,log_path):
    log_path=pathlib.Path(log_path)
    print('trainer command',json.dumps(cmd),flush=True)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        log.write('command='+json.dumps(cmd)+'\n'); log.flush()
        try: process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=CHILD_ENV)
        except Exception as exc:
            log.write('launch_error='+repr(exc)+'\n'); log.flush(); print('trainer launch failed',repr(exc),flush=True); raise
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait(); log.write(f'\nreturncode={returncode}\n'); log.flush()
    if returncode: raise subprocess.CalledProcessError(returncode,cmd)
POINTS=sorted(MODEL_CONTRACT['points'],key=lambda point:point['model_size'])
assert [point['model_size'] for point in POINTS]==[8,16,32,64,128],POINTS
assert len({point['model_tag'] for point in POINTS})==len(POINTS) and len({point['architecture_pair_id'] for point in POINTS})==len(POINTS)
assert all(point['model_family']=='lstm' for point in POINTS)
source_paths={'trainer_source_sha256':SCRIPT,'model_contract_source_sha256':CONTRACT_SCRIPT,'threshold_free_policy_source_sha256':f'{REPO}/formal_NN_training/common/threshold_free_policy.py'}
SWEEP=[]
for point in POINTS:
    out=f"{LOCAL_OUTPUT}/{point['model_tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
    for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
    cmd += ['--out-dir',out,'--model-family',point['model_family'],'--model-size',str(point['model_size']),'--pair-id',point['architecture_pair_id'],'--device','cuda','--seed',str(TRAINING['seed']),'--epochs',str(TRAINING['epochs']),'--chunk-len',str(TRAINING['chunk_len']),'--accumulate-chunks',str(TRAINING['accumulate_chunks']),'--learning-rate',str(TRAINING['learning_rate'])]
    log_path=TRAIN_LOG_DIR/f"{point['model_tag']}.stdout_stderr.log"
    print('\nTraining',point['model_tag'],flush=True); run_trainer_streamed(cmd,log_path)
    shutil.copy2(log_path,pathlib.Path(out,'trainer.stdout_stderr.log'))
    meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
    expected_meta={'model_tag':point['model_tag'],'model_family':point['model_family'],'model_size':point['model_size'],'architecture_pair_id':point['architecture_pair_id'],'matched_normal_prefetcher':POLICY,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_templates_used_by_neural_inference':False,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'inference_policy_hardcodes_used':False,'neural_degree_cap':None,'same_page_rule_used_by_neural_inference':False}
    projected=('operation','experiment_revision','model_revision','decoder_revision','runtime_feature_count','raw_runtime_feature_count','causal_runtime_feature_count','decoder_training_mode','delta_vocabulary_source','delta_vocabulary_max_exact','delta_other_escape','delta_other_decode_precision','exact_delta_representability_scope','hurdle_training_objective','hurdle_classes','separate_global_gate_used','separate_count_head_used','log_count_used','positive_count_training_objective','positive_count_support','terminal_stop_supervised_for_every_teacher_sequence','checkpoint_selection','decode_per_callback_resource_watchdog','decode_per_role_resource_watchdog','decode_resource_watchdog_behavior','decode_resource_watchdog_is_neural_degree_cap')
    expected_meta.update({key:MODEL_CONTRACT[key] for key in projected})
    bad={k:(meta.get(k),v) for k,v in expected_meta.items() if meta.get(k)!=v}; assert not bad,bad
    if 'run_id' in meta: assert meta['run_id']==RUN_ID
    assert meta['model_point_contract']==MODEL_CONTRACT and meta['decision_rule']==MODEL_CONTRACT['decoding_rule']
    assert meta['positive_count_decoding_rule']==MODEL_CONTRACT['positive_count_mode']
    assert meta['training_config']==TRAINING and meta['training_config_pinned_by_run_id'] is True
    assert meta['training_runtime_fields']==external_fields==meta['inference_runtime_fields']
    maximum=point['maximum_parameter_count']; assert 0<meta['parameter_count']==meta['realized_parameter_count']<=maximum
    assert meta['maximum_parameter_count']==maximum and meta['realized_parameter_count_matches_formula'] is True and meta['realized_parameter_count_within_maximum'] is True
    assert 0<meta['delta_vocabulary_exact_size']<=MODEL_CONTRACT['delta_vocabulary_max_exact']
    assert len(meta['delta_vocabulary_exact'])==meta['delta_vocabulary_exact_size']
    assert 1<=meta['selected_guard_epoch']<=TRAINING['epochs'] and meta['checkpoint_selection_roles']==['guard_metrics','TRAIN_loss_tiebreak_only']
    assert meta['guard_selection_composite_or_mean_used'] is False and len(meta['selected_guard_key'])==6
    determinism=MODEL_CONTRACT['determinism_contract']
    assert meta['training_device']=='cuda' and determinism['required_accelerator_name_contains'] in meta['training_device_name']
    assert meta['cublas_workspace_config']==determinism['cublas_workspace_config'] and meta['torch_deterministic_algorithms_enabled'] is True
    assert meta['cudnn_deterministic']==determinism['cudnn_deterministic'] and meta['cudnn_benchmark']==determinism['cudnn_benchmark'] and meta['float32_matmul_precision']==determinism['float32_matmul_precision']
    for key in MODEL_CONTRACT.get('required_source_hashes',source_paths):
        assert key in source_paths and meta[key]==hashlib.sha256(pathlib.Path(source_paths[key]).read_bytes()).hexdigest(),key
    encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
    assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
    SWEEP.append({key:meta[key] for key in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','maximum_parameter_count','selected_guard_epoch','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
sweep_manifest={'run_id':RUN_ID,'trace':TRACE,'policy':POLICY,'model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],'training_config':TRAINING,'external_input_fields':external_fields,'input_archive_provenance':INPUT_ARCHIVE_RECORD,'fresh_input_validation_manifest':VALIDATED_MANIFEST_PATH.name,'model_contract':MODEL_CONTRACT,'points':SWEEP}
pathlib.Path(f'{LOCAL_OUTPUT}/sweep_manifest.json').write_text(json.dumps(sweep_manifest,indent=2,sort_keys=True)+'\n')
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=pathlib.Path(DRIVE_ROOT,f'{RUN_ID}.colab_output.tar.gz')
OUTPUT_ARCHIVE_TEMP=OUTPUT_ARCHIVE.with_name(f'.{OUTPUT_ARCHIVE.name}.writing')
if OUTPUT_ARCHIVE_TEMP.exists(): OUTPUT_ARCHIVE_TEMP.unlink()
with tarfile.open(OUTPUT_ARCHIVE_TEMP,'w:gz') as archive:
    for item in sorted(pathlib.Path(OUTPUT_ROOT).iterdir(),key=lambda path:path.name): archive.add(item,arcname=item.name)
with tarfile.open(OUTPUT_ARCHIVE_TEMP,'r:gz') as archive: assert archive.getmembers()
os.replace(OUTPUT_ARCHIVE_TEMP,OUTPUT_ARCHIVE)
output_size=OUTPUT_ARCHIVE.stat().st_size
output_sha256=transfer.sha256_file(OUTPUT_ARCHIVE)
OUTPUT_PARTS_DIR=pathlib.Path(DRIVE_ROOT,'output_transfer_parts')
USE_MULTIPART_OUTPUT_FALLBACK=False  # Change only if the single browser download is impractical.
print('saved verified single output in Drive',OUTPUT_ARCHIVE,output_size,'bytes','sha256',output_sha256)
if not USE_MULTIPART_OUTPUT_FALLBACK:
    print('Downloading one archive (default).')
    files.download(str(OUTPUT_ARCHIVE))
else:
    if OUTPUT_PARTS_DIR.exists(): shutil.rmtree(OUTPUT_PARTS_DIR)
    OUTPUT_PARTS_DIR.mkdir(parents=True)
    output_manifest_path=transfer.split_archive(OUTPUT_ARCHIVE,OUTPUT_PARTS_DIR,overwrite=True)
    output_manifest=transfer.validate_parts(output_manifest_path,OUTPUT_PARTS_DIR)
    assert output_manifest['archive']['sha256']==transfer.sha256_file(OUTPUT_ARCHIVE)
    print('Using optional multipart output fallback; keep the manifest and every part together.')
    files.download(str(output_manifest_path))
    for part in output_manifest['parts']: files.download(str(OUTPUT_PARTS_DIR/part['name']))

The complete single output archive remains under the run-specific Google Drive folder and is the default browser download. Use the optional multipart branch only if that one-file transfer is impractical; if used, retain its JSON manifest and every numbered part for verification and reassembly. Trainer logs remain independently available in the run's `trainer_logs/` Drive directory even if a child process fails.